# CSC2042S Assignment 3: Logistic Regression

**Student Number:** `FFLADA001` &nbsp;&nbsp; **Name:** `Adam Foflonker`

### Pipeline:
1. **Data Loading & Preprocessing**
2. **Multinomial Logistic Regression Implementation**
3. **Training Setup**
4. **Hyperparameter Tuning**
5. **Feature Extraction**
6. **isiXhosa: Regularization & Handling Class Imbalance**
7. **Evaluation**
8. **Weight Analysis**
---

## Imports

In [ ]:
from __future__ import annotations

import os
import random
import re
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
import warnings

import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from sklearn.utils import resample
from collections import Counter


## Global Constants, State & Flags

In [ ]:
MASAKHANE_REPO_LOCAL_NAME: str = "data"
MASAKHANE_LANG_ENG: str = "eng"
MASAKHANE_LANG_XHO: str = "xho"

MAX_EPOCHS: int = 50
MAX_FEATURES: int = 10000
MIN_DF: int = 2

RANDOM_SEED: int = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

DEVICE: str = "cuda" if torch.cuda.is_available() else ("vulkan" if torch.is_vulkan_available() else "cpu")
print(f"Using PyTorch device: {DEVICE}.")

# Suppress ConvergenceWarning from scikit-learn
warnings.filterwarnings("ignore", category = UserWarning)


## 1) Data Loading & Preprocessing

Loads the MasakhaNEWS dataset for English and isiXhosa. This implementation combines the `headline` and `text` columns into a single feature. For preprocessing, a simple `basic_clean` function is applied. This function standardizes the text by:
- **Lowercasing**: Converts all text to lowercase to ensure that words like "Market" and "market" are treated as the same token.
- **Normalizing Whitespace**: Replaces newlines (`\n`) with spaces and removes excessive whitespaces.

In [ ]:
def find_local_masakhane_repo(root: str = '.') -> Optional[str]:
    """Finds the local directory containing the dataset."""
    for entry in os.listdir(root):
        if entry.lower().startswith(MASAKHANE_REPO_LOCAL_NAME) and os.path.isdir(os.path.join(root, entry)):
            return os.path.join(root, entry)
        
    return None

def load_masakhane_lang(repo_path: str, lang: str) -> Tuple[pd.Series, pd.Series, pd.Series, pd.Series, pd.Series, pd.Series, List[str]]:
    """Loads the MasakhaNEWS dataset for a specific language."""
    lang_dir = os.path.join(repo_path, "data", lang)
    if not os.path.isdir(lang_dir):
        raise FileNotFoundError(f"Language folder not found: {lang_dir}")

    def _read_tsv(name: str) -> pd.DataFrame:
        path = os.path.join(lang_dir, name)
        return pd.read_table(path, delimiter = '\t')

    df_train = _read_tsv("train.tsv")
    df_val = _read_tsv("dev.tsv")
    df_test = _read_tsv("test.tsv")

    with open(os.path.join(lang_dir, "labels.txt"), 'r', encoding = "utf-8") as f:
        labels_list = [line.strip() for line in f if line.strip()]

    X_train = (df_train["headline"].fillna('') + ' ' + df_train["text"].fillna('')).astype(str)
    X_val = (df_val["headline"].fillna('') + ' ' + df_val["text"].fillna('')).astype(str)
    X_test = (df_test["headline"].fillna('') + ' ' + df_test["text"].fillna('')).astype(str)

    label_to_idx = {label: i for i, label in enumerate(labels_list)}
    y_train = df_train["category"].map(label_to_idx).to_numpy()
    y_val = df_val["category"].map(label_to_idx).to_numpy()
    y_test = df_test["category"].map(label_to_idx).to_numpy()
    
    return X_train, y_train, X_val, y_val, X_test, y_test, labels_list

def basic_clean(text: str) -> str:
    """Lowercase and remove excessive whitespace."""
    t = text.lower()
    t = re.sub(r'\s+', ' ', t.replace('\n', ' ').strip())
    
    return t

# Locate repo and load data
repo = find_local_masakhane_repo()
if not repo:
    raise FileNotFoundError("Local masakhane-news repo not found! Please run 'setup.py' to clone it.")

print(f"Found local repository at: {repo}\n")

X_train_eng_raw, y_train_eng, X_val_eng_raw, y_val_eng, X_test_eng_raw, y_test_eng, labels_eng = load_masakhane_lang(repo, MASAKHANE_LANG_ENG)
X_train_xho_raw, y_train_xho, X_val_xho_raw, y_val_xho, X_test_xho_raw, y_test_xho, labels_xho = load_masakhane_lang(repo, MASAKHANE_LANG_XHO)

# Preprocess
X_train_eng = X_train_eng_raw.apply(basic_clean)
X_val_eng = X_val_eng_raw.apply(basic_clean)
X_test_eng = X_test_eng_raw.apply(basic_clean)

X_train_xho = X_train_xho_raw.apply(basic_clean)
X_val_xho = X_val_xho_raw.apply(basic_clean)
X_test_xho = X_test_xho_raw.apply(basic_clean)

print("English dataset loaded and preprocessed:")
print(f"  Train: X:{X_train_eng.shape}, Y:{y_train_eng.shape}")
print(f"  Val:   X:{X_val_eng.shape}, Y:{y_val_eng.shape}")
print(f"  Test:  X:{X_test_eng.shape}, Y:{y_test_eng.shape}")
print(f"  Labels: {labels_eng}\n")

print("isiXhosa dataset loaded and preprocessed:")
print(f"  Train: X:{X_train_xho.shape}, Y:{y_train_xho.shape}")
print(f"  Val:   X:{X_val_xho.shape}, Y:{y_val_xho.shape}")
print(f"  Test:  X:{X_test_xho.shape}, Y:{y_test_xho.shape}")
print(f"  Labels: {labels_xho}")


## 2) Multinomial Logistic Regression Implementation

Implements a class named `MultinomialLogisticRegression` which inherits from PyTorch's `nn.Module` class. This class serves as the **ML model** for performing logistic regression.

In [ ]:
class MultinomialLogisticRegression(nn.Module):
    """Simple multinomial logistic regression implemented in PyTorch."""
    def __init__(self, input_dim: int, num_classes: int):
        super().__init__()
        self.linear = nn.Linear(input_dim, num_classes) # Includes bias

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Return logits for input batch x (shape: batch x input_dim)."""
        return self.linear(x)

    def compute_probabilities(self, x: torch.Tensor) -> torch.Tensor:
        """Return softmax probabilities for input batch."""
        logits = self.forward(x)
        return torch.softmax(logits, dim = 1)

    def predict(self, x: torch.Tensor) -> torch.Tensor:
        """Return predicted class indices (tensor)."""
        probs = self.compute_probabilities(x)
        return torch.argmax(probs, dim = 1)


## 3) Training Setup

The main training loop. Training choices choices include:

- **Loss Function**: `nn.CrossEntropyLoss()` is used.
- **Optimizer**: `torch.optim.Adam` is chosen over standard SGD. Adam often converges faster and is less sensitive to the initial learning rate than SGD.
- **Stopping Criterion**: A fixed number of epochs is the primary stopping criterion.

In [ ]:
@dataclass
class TrainConfig:
    epochs: int = MAX_EPOCHS
    lr: float = 1e-2
    batch_size: int = 16
    l2_decay: float = 1e-4  # L2 Regularization (applied via optimizer)
    l1_lambda: float = 0.0 # L1 Regularization (applied manually in loss)
    device: str = DEVICE

def train_model(model: nn.Module, X_train: np.ndarray, y_train: np.ndarray, X_val: np.ndarray, y_val: np.ndarray, 
                cfg: TrainConfig) -> Dict:
    device = torch.device(cfg.device)
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr = cfg.lr, weight_decay = cfg.l2_decay) # Was SGD

    train_dataset = TensorDataset(torch.from_numpy(X_train).float(), torch.from_numpy(y_train).long())
    train_loader = DataLoader(train_dataset, batch_size = cfg.batch_size, shuffle = True)

    X_val_tensor = torch.from_numpy(X_val).float().to(device)
    y_val_tensor = torch.from_numpy(y_val).long().to(device)

    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    best_val_acc = -1.0
    best_state = None

    for epoch in range(cfg.epochs):
        model.train()
        running_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            
            # Add L1 regularization penalty if it was specified
            if cfg.l1_lambda > 0:
                l1_penalty = sum(p.abs().sum() for p in model.parameters())
                loss += cfg.l1_lambda * l1_penalty
            
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * xb.size(0)
            
        epoch_loss = running_loss / len(train_loader.dataset)

        # Validation
        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_tensor)
            val_loss = criterion(val_logits, y_val_tensor).item()
            preds = torch.argmax(val_logits, dim = 1).cpu().numpy()
            val_acc = accuracy_score(y_val, preds)

        history["train_loss"].append(epoch_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    return {"model": model, "history": history, "best_val_acc": best_val_acc}


## 4) Hyperparameter Tuning

The **learning rate** and **batch size** is tuned separately for each language. This is done using a *grid search* to find the best combination of the two based on validation accuracy.

In [ ]:
def run_single_experiment(lang_data, labels, config, feature_config):
    """Helper to run one full train-eval cycle."""
    X_train_raw, y_train, X_val_raw, y_val, _, _ = lang_data
    
    method = feature_config["method"]
    if (method == "count") or (method == "binary"):
        vectorizer = CountVectorizer(max_features = feature_config["max_features"], min_df = MIN_DF, binary = (method == "binary"))
    else:
        vectorizer = TfidfVectorizer(max_features = feature_config["max_features"], min_df = MIN_DF)
        
    X_train_vec = vectorizer.fit_transform(X_train_raw).toarray()
    X_val_vec = vectorizer.transform(X_val_raw).toarray()

    model = MultinomialLogisticRegression(input_dim = X_train_vec.shape[1], num_classes = len(labels))
    result = train_model(model, X_train_vec, y_train, X_val_vec, y_val, config)
    
    return result["best_val_acc"], result["history"]

def grid_search(lang, lang_data, labels):
    """Performs grid search for learning rate and batch size."""
    print(f"--- Starting Grid Search for {lang.upper()} ---")
    
    grid = {"lr": [1e-2, 1e-3, 1e-4], "batch_size": [8, 16, 32, 64]}
    feature_config = {"method": "tfidf", "max_features": MAX_FEATURES}
    best_acc = -1
    best_params = {}
    best_history = None
    results = []

    for lr in grid["lr"]:
        for batch_size in grid["batch_size"]:
            print(f"  Testing lr = {lr}, batch_size = {batch_size}...")
            cfg = TrainConfig(epochs = MAX_EPOCHS, lr = lr, batch_size = batch_size)
            val_acc, history = run_single_experiment(lang_data, labels, cfg, feature_config)
            results.append({"lr": lr, "batch_size": batch_size, "val_acc": val_acc})
            
            if val_acc > best_acc:
                best_acc = val_acc
                best_params = {"lr": lr, "batch_size": batch_size}
                best_history = history
    
    print(f"--- Grid Search for {lang.upper()} Complete ---")
    print(f"Best validation accuracy: {best_acc:.4f}")
    print(f"Best parameters: {best_params}")
    
    return best_params, best_history, pd.DataFrame(results)

def plot_history(history: Dict, title: str):
    """Plots training & validation loss and validation accuracy."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize = (15, 5))
    fig.suptitle(title, fontsize = 16)
    
    ax1.plot(history["train_loss"], label = "Train Loss", color = "red")
    ax1.plot(history["val_loss"], label = "Validation Loss", color = "purple")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.legend()
    ax1.grid(True)

    ax2.plot(history["val_acc"], label = "Validation Accuracy", color = "green")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy")
    ax2.legend()
    ax2.grid(True)
    
    plt.show()

# Run grid search for both languages
eng_data = (X_train_eng, y_train_eng, X_val_eng, y_val_eng, X_test_eng, y_test_eng)
xho_data = (X_train_xho, y_train_xho, X_val_xho, y_val_xho, X_test_xho, y_test_xho)

best_params_eng, history_eng, results_eng_df = grid_search("English", eng_data, labels_eng)
print("\nEnglish Grid Search Results:")
print(results_eng_df)

best_params_xho, history_xho, results_xho_df = grid_search("isiXhosa", xho_data, labels_xho)
print("\nisiXhosa Grid Search Results:")
print(results_xho_df)

# Plot the hyperparameter tuning analysis
plot_history(history_eng, "English: Best Model Training History")
plot_history(history_xho, "isiXhosa: Best Model Training History")


## 5) Feature Extraction

Compares three different methods for converting text into numerical vectors. The optimal hyperparameters found in the previous step are used here.

1.  **Bag-of-Words (Count)**: Represents text as a vector of word counts.
2.  **Binary Presence**: Similar to Bag-of-Words, but instead of counts, it only records whether a word is present (1) or not (0) in the document.
3.  **TF-IDF (Term Frequency-Inverse Document Frequency)**: This method weights word counts.

Each method is tested with different vocabulary sizes (`max_features`) to see how performance and training time are affected.

In [ ]:
def run_feature_extraction_experiments(lang, lang_data, labels, best_params):
    """Runs experiments for different feature extraction methods and vocab sizes."""
    print(f"--- Starting Feature Extraction Experiments for {lang.upper()} ---")
    
    feature_methods = ["count", "binary", "tfidf"]
    max_features_opts = [2000, 5000, MAX_FEATURES]
    results = []

    for method in feature_methods:
        for max_features in max_features_opts:
            print(f"  Testing method = {method}, max_features = {max_features}...")
            
            cfg = TrainConfig(epochs = MAX_EPOCHS, lr = best_params["lr"], batch_size = best_params["batch_size"])
            feature_config = {"method": method, "max_features": max_features}
            val_acc, _ = run_single_experiment(lang_data, labels, cfg, feature_config)
            results.append({"method": method, "max_features": max_features, "val_acc": val_acc})

    results_df = pd.DataFrame(results)
    best_result = results_df.loc[results_df["val_acc"].idxmax()]
    
    print(f"--- Feature Experiments for {lang.upper()} Complete ---")
    print("Results:")
    print(results_df)
    print(f"\nBest feature config: method = '{best_result["method"]}', max_features = {int(best_result["max_features"])} with accuracy {best_result["val_acc"]:.4f}\n")
    
    return {"method": best_result["method"], "max_features": int(best_result["max_features"])}

best_feature_eng = run_feature_extraction_experiments("English", eng_data, labels_eng, best_params_eng)
best_feature_xho = run_feature_extraction_experiments("isiXhosa", xho_data, labels_xho, best_params_xho)


## 6) isiXhosa: Regularization & Handling Class Imbalance

The isiXhosa dataset is smaller and more imbalanced than the English one, making it more prone to overfitting and biased towards majority classes. Two approaches to mitigate this have been investigated:

1.  **Regularization**: Add a penalty term to the loss function to discourage large model weights, which helps prevent overfitting.
    - **L2 Regularization (Weight Decay)**: Penalizes the squared magnitude of the weights.
    - **L1 Regularization**: Penalizes the absolute value of the weights. It can lead to sparse weights (some become exactly zero), effectively performing feature selection.

2.  **Handling Class Imbalance**: Rebalance the training data to ensure the model doesn't ignore minority classes.
    - **Upsampling Minority Classes**: Randomly duplicates samples from the minority classes until they have as many samples as the majority class.
    - **Downsampling Majority Classes**: Randomly removes samples from the majority classes until they have as many samples as the minority class.

In [ ]:
# Util function
def get_class_counts(y: np.ndarray) -> Dict[int, int]:
    return dict(Counter(y.tolist()))

def upsample_minority(X: np.ndarray, y: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Upsamples minority classes to match the majority class count."""
    df = pd.DataFrame({"text_index": np.arange(len(y)), "label": y})
    max_count = df["label"].value_counts().max()
    df_upsampled = pd.concat([resample(group, replace = True, n_samples = max_count, random_state = RANDOM_SEED) 
                              for _, group in df.groupby("label")])
    indices = df_upsampled["text_index"].values
    
    return X[indices], y[indices]

def downsample_majority(X: np.ndarray, y: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Downsamples majority classes to match the minority class count."""
    df = pd.DataFrame({"text_index": np.arange(len(y)), "label": y})
    min_count = df["label"].value_counts().min()
    df_downsampled = pd.concat([resample(group, replace = False, n_samples = min_count, random_state = RANDOM_SEED)
                                for _, group in df.groupby("label")])
    indices = df_downsampled["text_index"].values

    return X[indices], y[indices]

def run_xhosa_special_experiments():
    print("--- Starting isiXhosa Special Experiments ---")
    X_train_raw, y_train, X_val_raw, y_val, _, _ = xho_data
    results = []

    # Vectorize the data once using the best feature extractor
    vectorizer = TfidfVectorizer(max_features = best_feature_xho["max_features"], min_df = MIN_DF)
    X_train_vec = vectorizer.fit_transform(X_train_raw).toarray()
    X_val_vec = vectorizer.transform(X_val_raw).toarray()

    # Baseline (from the tuned hyperparameters)
    base_cfg = TrainConfig(epochs = MAX_EPOCHS, lr = best_params_xho["lr"], batch_size = best_params_xho["batch_size"])
    model = MultinomialLogisticRegression(input_dim = X_train_vec.shape[1], num_classes = len(labels_xho))
    base_res = train_model(model, X_train_vec, y_train, X_val_vec, y_val, base_cfg)
    results.append({"technique": "Baseline", "param": "N/A", "val_acc": base_res["best_val_acc"]})

    # L1 and L2 Regularization
    for l1_lambda in [1e-4, 1e-5]:
        cfg = TrainConfig(**best_params_xho, epochs = MAX_EPOCHS, l1_lambda = l1_lambda, l2_decay = 0)
        model = MultinomialLogisticRegression(input_dim = X_train_vec.shape[1], num_classes = len(labels_xho))
        res = train_model(model, X_train_vec, y_train, X_val_vec, y_val, cfg)
        results.append({"technique": "L1", "param": l1_lambda, "val_acc": res["best_val_acc"]})
        
    for l2_decay in [1e-3, 1e-4]:
        cfg = TrainConfig(**best_params_xho, epochs = MAX_EPOCHS, l2_decay = l2_decay, l1_lambda = 0)
        model = MultinomialLogisticRegression(input_dim = X_train_vec.shape[1], num_classes = len(labels_xho))
        res = train_model(model, X_train_vec, y_train, X_val_vec, y_val, cfg)
        results.append({"technique": "L2", "param": l2_decay, "val_acc": res["best_val_acc"]})

    # Class Imbalance
    print("Original class distribution:", get_class_counts(y_train))
    
    X_train_up, y_train_up = upsample_minority(X_train_vec, y_train)
    print("Upsampled class distribution:", get_class_counts(y_train_up))
    
    X_train_down, y_train_down = downsample_majority(X_train_vec, y_train)
    print("Downsampled class distribution:", get_class_counts(y_train_down))
    
    model_up = MultinomialLogisticRegression(input_dim = X_train_up.shape[1], num_classes = len(labels_xho))
    res_up = train_model(model_up, X_train_up, y_train_up, X_val_vec, y_val, base_cfg)
    results.append({"technique": "Upsampling", "param": "N/A", "val_acc": res_up["best_val_acc"]})
    
    model_down = MultinomialLogisticRegression(input_dim=X_train_down.shape[1], num_classes=len(labels_xho))
    res_down = train_model(model_down, X_train_down, y_train_down, X_val_vec, y_val, base_cfg)
    results.append({"technique": "Downsampling", "param": "N/A", "val_acc": res_down["best_val_acc"]})

    results_df = pd.DataFrame(results)
    print("\n--- isiXhosa Special Experiments Complete ---")
    print(results_df)
    
    best_technique = results_df.loc[results_df["val_acc"].idxmax()]
    print(f"\nBest technique for isiXhosa: {best_technique["technique"]} (param: {best_technique["param"]}) with accuracy {best_technique["val_acc"]:.4f}")
    
    return best_technique.to_dict()

best_technique_xho = run_xhosa_special_experiments()


## 7) Evaluation

Takes the best models — determined by the optimal hyperparameters, feature extraction method, and (for isiXhosa) any special techniques, and evaluates their performance on the unseen **test set**.

This implementation reports:
- **Accuracy**
- **Precision, Recall, F1-score**
  - **Micro-averaged**: Aggregates the contributions of all classes to compute the average metric.
  - **Macro-averaged**: Calculates the metric independently for each class and then takes the average.

A **Confusion Matrix** is also plotted for each language to visualize which categories are being confused with one another.

In [ ]:
def evaluate_final_model(model: nn.Module, X_test: np.ndarray, y_test: np.ndarray, label_names: List[str]):
    """Evaluates the model on the test set and prints report."""
    model.eval()
    with torch.no_grad():
        X_t = torch.from_numpy(X_test).float().to(DEVICE)
        preds = model.predict(X_t).cpu().numpy()

    acc = accuracy_score(y_test, preds)
    precision_micro, recall_micro, f1_micro, _ = precision_recall_fscore_support(y_test, preds, average='micro')
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(y_test, preds, average='macro')
    
    print(f"Accuracy: {acc:.4f}")
    print("\n--- Micro-Averaged Metrics ---")
    print(f"  Precision: {precision_micro:.4f}")
    print(f"  Recall:    {recall_micro:.4f}")
    print(f"  F1-Score:  {f1_micro:.4f}")
    print("\n--- Macro-Averaged Metrics ---")
    print(f"  Precision: {precision_macro:.4f}")
    print(f"  Recall:    {recall_macro:.4f}")
    print(f"  F1-Score:  {f1_macro:.4f}")

    # Confusion Matrix
    cm = confusion_matrix(y_test, preds)
    plt.figure(figsize = (8, 6))
    sns.heatmap(cm, annot = True, fmt = 'd', xticklabels = label_names, yticklabels = label_names)
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title("Confusion Matrix")
    plt.show()
    
    return {"accuracy": acc, "f1_macro": f1_macro}

def train_and_evaluate_best_model(lang, lang_data, labels, best_params, best_feature, best_technique=None):
    """Trains the final best model on train+val and evaluates on test."""
    print(f"\n--- Final Evaluation for {lang.upper()} ---")
    
    X_train_raw, y_train, X_val_raw, y_val, X_test_raw, y_test = lang_data
    
    # Combine train and validation sets for final training
    X_train_full_raw = pd.concat([X_train_raw, X_val_raw], ignore_index = True)
    y_train_full = np.concatenate([y_train, y_val])

    # Vectorizer
    method = best_feature["method"]
    if (method == "count") or (method == "binary"):
        vectorizer = CountVectorizer(max_features = best_feature["max_features"], min_df = MIN_DF, binary = (method == "binary"))
    else:
        vectorizer = TfidfVectorizer(max_features = best_feature["max_features"], min_df = MIN_DF)
    
    X_train_full_vec = vectorizer.fit_transform(X_train_full_raw).toarray()
    X_test_vec = vectorizer.transform(X_test_raw).toarray()
    
    # Handle sampling technique for isiXhosa if required
    if best_technique and (best_technique["technique"] == "Upsampling"):
        X_train_full_vec, y_train_full = upsample_minority(X_train_full_vec, y_train_full)
    elif best_technique and (best_technique["technique"] == "Downsampling"):
        X_train_full_vec, y_train_full = downsample_majority(X_train_full_vec, y_train_full)
        
    # Training config
    cfg = TrainConfig(epochs = MAX_EPOCHS, lr = best_params["lr"], batch_size = best_params["batch_size"])
    
    if best_technique and (best_technique["technique"] == "L1"):
        cfg.l1_lambda = best_technique["param"]
        
    if best_technique and best_technique["technique"] == "L2":
        cfg.l2_decay = best_technique["param"]

    # Final training and evaluation
    model = MultinomialLogisticRegression(input_dim = X_train_full_vec.shape[1], num_classes = len(labels))
    
    # We use a dummy validation set here since we're training on the full data
    # The model state at the final epoch will be used.
    final_result = train_model(model, X_train_full_vec, y_train_full, X_train_full_vec[:10], y_train_full[:10], cfg)
    final_model = final_result["model"]
    evaluate_final_model(final_model, X_test_vec, y_test, labels)
    
    return final_model, vectorizer

# Run final evaluation for English and isiXhosa
final_model_eng, vectorizer_eng = train_and_evaluate_best_model("English", eng_data, labels_eng, best_params_eng, best_feature_eng)
final_model_xho, vectorizer_xho = train_and_evaluate_best_model("isiXhosa", xho_data, labels_xho, best_params_xho, best_feature_xho, 
                                                                best_technique_xho)


## 8) Weight Analysis

Inspect the top `40` learned weights of the English model.

In [ ]:
def top_features_per_class(model: MultinomialLogisticRegression, vectorizer, labels: List[str], top_k: int = 40):
    """Displays the top-k features (words) for each class based on learned weights."""
    W = model.linear.weight.detach().cpu().numpy()
    feature_names = vectorizer.get_feature_names_out()
    
    for i, label in enumerate(labels):
        row = W[i]
        top_idx = np.argsort(row)[-top_k:][::-1]
        top_features = [(feature_names[j], f"{row[j]:.3f}") for j in top_idx]
        
        print(f"--- Top features for class: '{label}' ---")
        print(top_features)
        print()

print("\nAnalyzing learned weights for the final English model:\n")
top_features_per_class(final_model_eng, vectorizer_eng, labels_eng)
